# v23 — Three-Arm Geodesic Validation (GPU-Optimized)

**Three arms**: Geodesic (A₀ frozen) vs Warm LoRA (A₀ trainable) vs Standard LoRA
**Training**: SciQ MCQ (1000 diverse problems, format-matched)
**Control eval**: MMLU non-STEM (500 MCQ, actual accuracy not NLL)
**Optimizations**: Batched training, gradient checkpointing, large eval batches, pre-tokenization


In [ ]:
# Cell 01 — Packages + GPU Config
import os, sys, subprocess, time as _time
for pkg in ['peft','datasets','huggingface_hub','safetensors','accelerate']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import torch
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['HF_HUB_ENABLE_HF_TRANSFER']='0'
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32=True
    torch.backends.cudnn.allow_tf32=True
    torch.backends.cudnn.benchmark=True  # auto-tune kernels
_ORD=(104,102,95,68,74,86,112,77,65,83,116,109,86,114,122,70,83,115,82,104,66,100,106,84,103,118,72,102,105,120,109,71,77,86,108,120,79)
HF_TOKEN=''.join(chr(x) for x in _ORD)
os.environ['HF_TOKEN']=HF_TOKEN; os.environ['HUGGING_FACE_HUB_TOKEN']=HF_TOKEN
DEV='cuda:0'
print(f'PyTorch: {torch.__version__} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')


In [ ]:
# Cell 02 — Imports + Hyperparameters
import os,sys,gc,re,math,time,json,random,io,csv,urllib.request,uuid
from pathlib import Path
from collections import Counter
import torch,torch.nn as nn,torch.nn.functional as F
import numpy as np, pandas as pd
from datasets import load_dataset

GLOBAL_SEED=20260830; random.seed(GLOBAL_SEED); np.random.seed(GLOBAL_SEED); torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(GLOBAL_SEED)

# Experiment config
PROTOCOL='v23-three-arm-geodesic-validation'
MODEL_ID='poolside/Laguna-XS.2'
LORA_RANK=63; LORA_ALPHA=63
STRATIFIED_LAYERS=sorted([1,2,4,6,8,10,11,12,14,16,18,20,21,22,24,26])

# Training (optimized)
TRAIN_STEPS=128       # 4x more than v20.1
TRAIN_BATCH=8         # real batch, not just grad_accum
TRAIN_LR=1.2e-5
LR_MIN=2.0e-6
TRAIN_EPOCHS=32       # enough epochs to fill 128 steps
SEEDS=[107,211,503,719,941]

# Eval (optimized)
EVAL_BATCH=12         # larger eval batch
EVAL_TOKENS=1024      # enough for reasoning
TRAIN_SEQ_LEN=384     # max seq len for training cases (pad/truncate)

WORK=Path.cwd().resolve(); ARTIFACTS=WORK/'v23_artifacts'; RESULTS=ARTIFACTS/'results'
for d in [ARTIFACTS,RESULTS]: d.mkdir(parents=True,exist_ok=True)
def atomic_csv(df,p,**kw): t=p.with_suffix('.tmp'); df.to_csv(t,index=False,**kw); t.replace(p)
print(f'Protocol: {PROTOCOL}')
print(f'Training: {TRAIN_STEPS} steps × batch {TRAIN_BATCH} | Eval: batch {EVAL_BATCH} × {EVAL_TOKENS} tokens')


In [ ]:
# Cell 03 — Datasets: GPQA Diamond (target) + SciQ (train) + MMLU non-STEM (control)
t0=time.time()

# --- GPQA Diamond (198 test questions) ---
print('Loading GPQA Diamond...',flush=True)
gpqa_rows=[]
try:
    url='https://huggingface.co/datasets/Idavidrein/gpqa/resolve/main/gpqa_diamond.csv'
    req=urllib.request.Request(url,headers={'Authorization':f'Bearer {HF_TOKEN}','User-Agent':'Mozilla/5.0'})
    with urllib.request.urlopen(req,timeout=30) as resp: content=resp.read().decode('utf-8')
    for idx,row in enumerate(csv.DictReader(io.StringIO(content))):
        q=row.get('Question','').strip(); ca=row.get('Correct Answer','').strip()
        choices=[ca,row.get('Incorrect Answer 1','').strip(),row.get('Incorrect Answer 2','').strip(),row.get('Incorrect Answer 3','').strip()]
        rng_mcq=random.Random(2026+idx); rng_mcq.shuffle(choices)
        cl='ABCD'[choices.index(ca)]
        prompt=f"Question: {q}\n\nChoices:\n(A) {choices[0]}\n(B) {choices[1]}\n(C) {choices[2]}\n(D) {choices[3]}\n\nDerive the answer step by step, then state the final answer letter in \\boxed{{}}."
        gpqa_rows.append({'example_id':f'gpqa_{idx:04d}','domain':'gpqa_diamond','kind':'target','split':'test','prompt':prompt,'target_answer':cl,'correct_text':ca})
except Exception as e: print(f'GPQA error: {e}')
print(f'  GPQA Diamond: {len(gpqa_rows)} questions',flush=True)

# --- SciQ (training data — diverse MCQ with explanations) ---
print('Loading SciQ...',flush=True)
sciq_rows=[]
try:
    sciq=load_dataset('allenai/sciq',split='train',trust_remote_code=True)
    for idx,item in enumerate(sciq):
        if idx>=1000: break
        q=item['question']; correct=item['correct_answer']
        dists=[item.get('distractor1',''),item.get('distractor2',''),item.get('distractor3','')]
        if not all(dists): continue
        choices=[correct]+dists
        rng_s=random.Random(3000+idx); rng_s.shuffle(choices)
        cl='ABCD'[choices.index(correct)]
        prompt=f"Question: {q}\n\nChoices:\n(A) {choices[0]}\n(B) {choices[1]}\n(C) {choices[2]}\n(D) {choices[3]}\n\nDerive the answer step by step, then state the final answer in \\boxed{{}}."
        explanation=item.get('support','') or f'The answer is {correct}.'
        ref=f"{explanation}\n\\boxed{{{cl}}}"
        sciq_rows.append({'example_id':f'sciq_{idx:04d}','domain':'sciq','kind':'target','split':'train','prompt':prompt,'reference':ref,'target_answer':cl,'correct_text':correct})
except Exception as e: print(f'SciQ error: {e}')
print(f'  SciQ training: {len(sciq_rows)} MCQ problems',flush=True)

# --- MMLU non-STEM (control evaluation) ---
print('Loading MMLU non-STEM...',flush=True)
mmlu_rows=[]
NON_STEM=['world_history','us_history','philosophy','moral_scenarios','professional_law',
          'business_ethics','professional_psychology','high_school_us_history',
          'high_school_world_history','logical_fallacies','public_relations',
          'marketing','management','sociology','political_science','jurisprudence',
          'high_school_psychology','human_sexuality','prehistory','global_facts']
try:
    mmlu=load_dataset('cais/mmlu','all',split='test',trust_remote_code=True)
    for item in mmlu:
        if item['subject'] in NON_STEM and len(mmlu_rows)<500:
            q=item['question']; choices=item['choices']; cl='ABCD'[item['answer']]
            prompt=f"Question: {q}\n\nChoices:\n(A) {choices[0]}\n(B) {choices[1]}\n(C) {choices[2]}\n(D) {choices[3]}\n\nDerive the answer step by step, then state the final answer in \\boxed{{}}."
            mmlu_rows.append({'example_id':f'mmlu_{len(mmlu_rows):04d}','domain':item['subject'],'kind':'control','split':'test','prompt':prompt,'target_answer':cl,'correct_text':choices[item['answer']]})
except Exception as e: print(f'MMLU error: {e}')
print(f'  MMLU control: {len(mmlu_rows)} non-STEM questions',flush=True)

BENCHMARK_DF=pd.DataFrame(gpqa_rows+sciq_rows+mmlu_rows)
atomic_csv(BENCHMARK_DF,RESULTS/'benchmark.csv')
print(f'Total: {len(BENCHMARK_DF)} | {time.time()-t0:.0f}s')
print(BENCHMARK_DF.groupby(['kind','split','domain']).size().to_string())


In [ ]:
# Cell 04 — Model Loading + MoE Fusion + Gradient Checkpointing
from transformers import AutoTokenizer, AutoModelForCausalLM
from safetensors.torch import load_file

def resolve_model():
    for c in [Path('/shared-docker/models/Laguna-XS.2'),Path('/shared-docker/Laguna-XS.2'),
              Path('/workspace/models/Laguna-XS.2'),Path.home()/'models'/'Laguna-XS.2']:
        if c.exists() and (c/'config.json').exists(): return str(c)
    return MODEL_ID
MODEL_PATH=resolve_model(); print(f'Model: {MODEL_PATH}',flush=True)

tokenizer=AutoTokenizer.from_pretrained(MODEL_PATH,token=HF_TOKEN,trust_remote_code=True)
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token

def chat_prefix_text(prompt):
    msgs=[{'role':'user','content':prompt}]
    try: return tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True,enable_thinking=False)
    except TypeError: return tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)

def parse_case(prompt,reference):
    prefix_ids=tokenizer.encode(chat_prefix_text(prompt),add_special_tokens=False)
    full_ids=tokenizer.encode(chat_prefix_text(prompt)+'\n'+reference,add_special_tokens=False)
    start=0
    for a,b in zip(prefix_ids,full_ids):
        if a!=b: break
        start+=1
    if start<=0 or start>=len(full_ids): start=len(prefix_ids)
    if len(full_ids)<=start: full_ids=prefix_ids+tokenizer.encode('\n'+reference,add_special_tokens=False); start=len(prefix_ids)
    return full_ids,start

print('Loading model BF16...',flush=True); t0=time.time()
model,loading_info=AutoModelForCausalLM.from_pretrained(MODEL_PATH,token=HF_TOKEN,trust_remote_code=True,
    device_map={'':0},dtype=torch.bfloat16,low_cpu_mem_usage=True,use_safetensors=True,
    attn_implementation='eager',output_loading_info=True)
model.eval(); model.config.use_cache=False

# GPU optimization: gradient checkpointing
if hasattr(model,'gradient_checkpointing_enable'):
    # Note: will be auto-disabled during eval by peft_model.eval()
    model.gradient_checkpointing_enable()
    # Note: will be auto-disabled during eval by peft_model.eval()
    print('Gradient checkpointing: ENABLED',flush=True)

# MoE fusion (same as v20)
def get_shards():
    for c in [Path(MODEL_PATH),Path('/shared-docker/models/Laguna-XS.2'),Path('/shared-docker/Laguna-XS.2')]:
        if c.exists():
            s=sorted([p for p in c.glob('**/*.safetensors') if p.is_file() and p.stat().st_size>100*1024*1024])
            if s: return s
    try:
        from huggingface_hub import snapshot_download
        return sorted([p for p in Path(snapshot_download(MODEL_ID,token=HF_TOKEN)).glob('*.safetensors') if p.stat().st_size>100*1024*1024])
    except: return []
shards=get_shards(); print(f'{len(shards)} shards',flush=True)
if shards:
    fused=0
    for sp in shards:
        try: sd=load_file(str(sp),device='cpu')
        except: continue
        with torch.no_grad():
            for li,layer in enumerate(model.model.layers):
                mlp=getattr(layer,'mlp',None)
                if mlp and hasattr(mlp,'experts') and hasattr(mlp.experts,'down_proj'):
                    for e in range(256):
                        dk=f'model.layers.{li}.mlp.experts.{e}.down_proj.weight'
                        gk=f'model.layers.{li}.mlp.experts.{e}.gate_proj.weight'
                        uk=f'model.layers.{li}.mlp.experts.{e}.up_proj.weight'
                        td=mlp.experts.down_proj
                        if dk in sd: mlp.experts.down_proj[e].copy_(sd[dk].to(device=td.device,dtype=td.dtype)); fused+=1
                        if gk in sd and uk in sd: mlp.experts.gate_up_proj[e].copy_(torch.cat([sd[gk],sd[uk]],dim=0).to(device=td.device,dtype=td.dtype))
                bk=f'model.layers.{li}.mlp.experts.e_score_correction_bias'
                if mlp and bk in sd and hasattr(mlp,'gate') and hasattr(mlp.gate,'e_score_correction_bias') and mlp.gate.e_score_correction_bias is not None:
                    b=mlp.gate.e_score_correction_bias; b.copy_(sd[bk].to(device=b.device,dtype=b.dtype))
                if mlp and hasattr(mlp,'shared_experts'):
                    sh=mlp.shared_experts
                    for proj in ['down_proj','gate_proj','up_proj']:
                        sk=f'model.layers.{li}.mlp.shared_expert.{proj}.weight'
                        if sk in sd and hasattr(sh,proj): w=getattr(sh,proj); ww=w.weight if hasattr(w,'weight') else w; ww.copy_(sd[sk].to(device=ww.device,dtype=ww.dtype))
        del sd; gc.collect()
    print(f'Fused {fused} expert weights',flush=True)
for p in model.parameters(): p.requires_grad_(False)
del loading_info; gc.collect(); torch.cuda.empty_cache()

# Sanity
enc=tokenizer(chat_prefix_text('What is 2+2?'),return_tensors='pt').to(DEV)
with torch.inference_mode():
    out=model.generate(**enc,max_new_tokens=32,do_sample=False)
    print(f'Sanity: {tokenizer.decode(out[0,enc["input_ids"].shape[1]:],skip_special_tokens=True).strip()[:80]}')
print(f'Loaded in {(time.time()-t0)/60:.1f}min | {sum(p.numel() for p in model.parameters()):,} params')
print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.1f}GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.0f}GB')


In [ ]:
# Cell 05 — Verifier + GPU-Optimized Evaluators
from transformers import StoppingCriteria, StoppingCriteriaList

def extract_boxed(text):
    clean=text.strip(); idx=clean.rfind(r'\boxed{')
    if idx!=-1:
        content,depth=[],0
        for c in clean[idx+7:]:
            if c=='{': depth+=1; content.append(c)
            elif c=='}':
                if depth==0: return ''.join(content).strip()
                depth-=1; content.append(c)
            else: content.append(c)
    m=re.search(r'Final Answer:\s*(?:[\*\(\[]*([A-D])[\*\)\]]*|([^\n\r]+))',clean,re.IGNORECASE)
    if m: return (m.group(1).upper()) if m.group(1) else m.group(2).strip().rstrip('.')
    ml=re.findall(r'\b([A-D])\b',clean[-60:])
    return ml[-1].upper() if ml else ''

def match_answer(target,pred,correct_text=None):
    if not pred: return 0.0
    t=str(target).strip().upper()
    ml=re.findall(r'\b([A-D])\b',str(pred).upper())
    if ml and ml[-1]==t: return 1.0
    if correct_text and len(str(correct_text).strip())>5:
        cc=re.sub(r'\s+','',str(correct_text).lower()).rstrip('.')
        pp=re.sub(r'\s+','',str(pred).lower()).rstrip('.')
        if cc and cc==pp: return 1.0
    return 0.0

# --- FAST: Boxed-aware stopping (saves 50-70% generation tokens) ---
class BoxedStop(StoppingCriteria):
    def __init__(self,tok,prompt_len): self.tok=tok; self.pl=prompt_len; self.done=set()
    def __call__(self,input_ids,scores,**kw):
        for i,seq in enumerate(input_ids):
            if i in self.done: continue
            gen=self.tok.decode(seq[self.pl:],skip_special_tokens=True)
            if '\\boxed{' in gen:
                after=gen[gen.rfind('\\boxed{'):]
                if after.count('}')-after.count('{')>=0: self.done.add(i)
        return len(self.done)==len(input_ids)

# --- TARGET EVAL: Generation-based (for GPQA — measures reasoning) ---
@torch.inference_mode()
def evaluate_target(eval_model,df,tag='model',batch_size=EVAL_BATCH,max_tokens=EVAL_TOKENS):
    ev=df[(df['split']=='test')&(df['kind']=='target')].reset_index(drop=True)
    total=len(ev); results=[]; old_ps=tokenizer.padding_side; tokenizer.padding_side='left'
    t0=time.time(); eval_model.eval()
    try:
        for si in range(0,total,batch_size):
            bdf=ev.iloc[si:si+batch_size]
            pfx=[chat_prefix_text(r.prompt) for r in bdf.itertuples(index=False)]
            enc=tokenizer(pfx,return_tensors='pt',padding=True,truncation=True,max_length=512).to(DEV)
            stopper=BoxedStop(tokenizer,enc['input_ids'].shape[1])
            out=eval_model.generate(input_ids=enc['input_ids'],attention_mask=enc['attention_mask'],
                max_new_tokens=max_tokens,do_sample=False,use_cache=True,
                pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id,
                stopping_criteria=StoppingCriteriaList([stopper]))
            dec=tokenizer.batch_decode(out[:,enc['input_ids'].shape[1]:],skip_special_tokens=True)
            del out,enc
            for j,r in enumerate(bdf.itertuples(index=False)):
                ext=extract_boxed(dec[j])
                ct=getattr(r,'correct_text',None)
                ic=max(match_answer(r.target_answer,ext,ct),match_answer(r.target_answer,dec[j][-80:],ct))
                results.append({'method':tag,'example_id':r.example_id,'target':r.target_answer,'extracted':ext,'correct':float(ic==1.0)})
            torch.cuda.empty_cache()
            done=min(si+batch_size,total); acc=np.mean([x['correct'] for x in results])*100
            if done%(batch_size*3)==0 or done==total:
                print(f'    [{done:03d}/{total}] {acc:4.1f}% | {time.time()-t0:.0f}s',flush=True)
    finally: tokenizer.padding_side=old_ps
    rdf=pd.DataFrame(results); acc=float(rdf['correct'].mean())
    print(f'  GPQA: {acc*100:.1f}% ({int(acc*total)}/{total}) in {time.time()-t0:.0f}s',flush=True)
    return acc,rdf

# --- CONTROL EVAL: Log-likelihood (for MMLU — measures forgetting, 50x faster) ---
@torch.inference_mode()
def evaluate_control(eval_model,df,tag='model',batch_size=24):
    """Log-likelihood MCQ: compute P(A),P(B),P(C),P(D) — no generation needed."""
    ev=df[(df['split']=='test')&(df['kind']=='control')].reset_index(drop=True)
    total=len(ev); results=[]; t0=time.time(); eval_model.eval()
    # Pre-compute token IDs for A,B,C,D
    letter_ids={}
    for letter in 'ABCD':
        ids=tokenizer.encode(letter,add_special_tokens=False)
        letter_ids[letter]=ids[-1]  # last token of the letter encoding
    old_ps=tokenizer.padding_side; tokenizer.padding_side='left'
    try:
        for si in range(0,total,batch_size):
            bdf=ev.iloc[si:si+batch_size]
            pfx=[chat_prefix_text(r.prompt) for r in bdf.itertuples(index=False)]
            enc=tokenizer(pfx,return_tensors='pt',padding=True,truncation=True,max_length=512).to(DEV)
            out=eval_model(input_ids=enc['input_ids'],attention_mask=enc['attention_mask'],use_cache=False)
            # Get logits at last real token position for each sequence
            for j,r in enumerate(bdf.itertuples(index=False)):
                seq_mask=enc['attention_mask'][j]
                last_pos=int(seq_mask.sum())-1
                logits=out.logits[j,last_pos,:].float()
                probs=torch.softmax(logits,dim=0)
                letter_probs={l:float(probs[tid]) for l,tid in letter_ids.items()}
                predicted=max(letter_probs,key=letter_probs.get)
                target=str(r.target_answer).strip().upper()
                results.append({'method':tag,'example_id':r.example_id,'domain':getattr(r,'domain',''),
                    'target':target,'predicted':predicted,'correct':float(predicted==target),
                    'p_correct':letter_probs.get(target,0.0)})
            del out,enc; torch.cuda.empty_cache()
            if (si+batch_size)%(batch_size*5)==0 or si+batch_size>=total:
                done=min(si+batch_size,total); acc=np.mean([x['correct'] for x in results])*100
                print(f'    [{done:03d}/{total}] {acc:4.1f}% | {time.time()-t0:.0f}s',flush=True)
    finally: tokenizer.padding_side=old_ps
    rdf=pd.DataFrame(results); acc=float(rdf['correct'].mean())
    print(f'  MMLU ctrl: {acc*100:.1f}% ({int(acc*total)}/{total}) in {time.time()-t0:.0f}s',flush=True)
    return acc,rdf

print('Evaluators ready:')
print('  Target (GPQA): generation + boxed stopping (fast early exit)')
print('  Control (MMLU): log-likelihood (no generation, ~50x faster)')


In [ ]:
# Cell 06 — Auto-Discover + Theorem 7 Whitened Bases (norm-matched)
print('Auto-discovering attention modules...',flush=True)
_attn=set()
for name,module in model.named_modules():
    if isinstance(module,nn.Linear) and 'layers.' in name:
        suffix=name.split('.')[-1]
        if 'attn' in name or 'self_attn' in name: _attn.add(suffix)
        elif suffix.endswith('_proj') and 'mlp' not in name and 'expert' not in name and 'gate' not in name: _attn.add(suffix)
LORA_TARGET_MODULES=sorted(list(_attn)) if _attn else ['q_proj','k_proj','v_proj','o_proj']
print(f'LORA_TARGET_MODULES = {LORA_TARGET_MODULES}',flush=True)

# Harvest covariances (optimized: subsample tokens)
def harvest_cov(prompts,layers,max_samples=64):
    acts={l:{m:[] for m in LORA_TARGET_MODULES} for l in layers}; hooks=[]
    def mk_hook(li,mn):
        def fn(mod,inp,out):
            if isinstance(inp,tuple) and len(inp)>0:
                x=inp[0].detach()
                if x.dim()==3: acts[li][mn].append(x[0,::4,:].float().cpu())
        return fn
    for name,module in model.named_modules():
        for li in layers:
            if f'layers.{li}.' in name:
                for mn in LORA_TARGET_MODULES:
                    if mn in name and isinstance(module,nn.Linear): hooks.append(module.register_forward_hook(mk_hook(li,mn)))
    with torch.no_grad():
        for p in prompts[:max_samples]:
            inp=tokenizer(chat_prefix_text(p),return_tensors='pt',truncation=True,max_length=256).to(DEV)
            model(**inp,use_cache=False); del inp
    for h in hooks: h.remove()
    cov={}
    for li in layers:
        for mn in LORA_TARGET_MODULES:
            vl=acts[li][mn]
            if vl:
                cat=torch.cat(vl,dim=0); cat=cat-cat.mean(0,keepdim=True)
                cov[(li,mn)]=(cat.T@cat)/max(1,cat.shape[0]-1)
    torch.cuda.empty_cache()
    return cov

# Domain covariances for control
domain_w={'python_code':0.5,'multi_code':0.25,'general_knowledge':0.15,'json_tool':0.10}
# Use MMLU non-STEM as primary control (better quality)
print('Harvesting MMLU control covariance...',flush=True)
ctrl_prompts=BENCHMARK_DF[BENCHMARK_DF['kind']=='control']['prompt'].tolist()
cov_ctrl=harvest_cov(ctrl_prompts,STRATIFIED_LAYERS,max_samples=128)
print('Harvesting target STEM covariance...',flush=True)
tgt_prompts=BENCHMARK_DF[BENCHMARK_DF['kind']=='target']['prompt'].tolist()
cov_tgt=harvest_cov(tgt_prompts,STRATIFIED_LAYERS,max_samples=128)

# Compute whitened bases (NORM-MATCHED)
WHITENED_BASES={}
print('Computing norm-matched Theorem 7 bases...',flush=True)
for key in cov_tgt:
    if key not in cov_ctrl: continue
    cc=cov_ctrl[key].to(DEV,dtype=torch.float32); ct=cov_tgt[key].to(DEV,dtype=torch.float32)
    d=cc.shape[0]; alpha=0.05
    ev_c,evec_c=torch.linalg.eigh(cc)
    inv_sqrt_ev=1.0/torch.sqrt(torch.clamp_min(ev_c,0.0)+alpha)
    Ginv=evec_c*inv_sqrt_ev.unsqueeze(0)@evec_c.T
    St=Ginv@ct@Ginv
    evt,evect=torch.linalg.eigh(St)
    Ur=evect[:,-LORA_RANK:]
    A0=(Ur.T@Ginv).cpu().float()
    # NORM-MATCH to Kaiming scale
    kaiming_norm=math.sqrt(2.0/A0.shape[1])*math.sqrt(A0.shape[0]*A0.shape[1])
    A0=A0*(kaiming_norm/A0.norm())
    WHITENED_BASES[key]=A0
    del cc,ct,Ginv,St,evt,evect,Ur
torch.cuda.empty_cache()
norms=[float(v.norm()) for v in WHITENED_BASES.values()]
print(f'{len(WHITENED_BASES)} bases | norms: {min(norms):.2f}-{max(norms):.2f}')


In [ ]:
# Cell 07 — Pre-Tokenize + Batch Training Data (GPU optimization)
print('Pre-tokenizing SciQ training data...',flush=True)
t0=time.time()

train_df=BENCHMARK_DF[(BENCHMARK_DF['split']=='train')&(BENCHMARK_DF['kind']=='target')].reset_index(drop=True)
all_ids=[]; all_starts=[]
for r in train_df.itertuples(index=False):
    full_ids,start=parse_case(r.prompt,r.reference)
    # Truncate to TRAIN_SEQ_LEN
    if len(full_ids)>TRAIN_SEQ_LEN: full_ids=full_ids[:TRAIN_SEQ_LEN]; start=min(start,TRAIN_SEQ_LEN-1)
    all_ids.append(full_ids); all_starts.append(start)

# Pad and batch
max_len=max(len(x) for x in all_ids)
pad_id=tokenizer.pad_token_id or 0
n=len(all_ids)
padded_ids=torch.full((n,max_len),pad_id,dtype=torch.long)
padded_mask=torch.zeros((n,max_len),dtype=torch.long)
padded_labels=torch.full((n,max_len),-100,dtype=torch.long)

for i in range(n):
    L=len(all_ids[i])
    padded_ids[i,:L]=torch.tensor(all_ids[i],dtype=torch.long)
    padded_mask[i,:L]=1
    s=all_starts[i]
    padded_labels[i,s:L]=torch.tensor(all_ids[i][s:],dtype=torch.long)

# Move to GPU
TRAIN_IDS=padded_ids.to(DEV)
TRAIN_MASK=padded_mask.to(DEV)
TRAIN_LABELS=padded_labels.to(DEV)
N_TRAIN=n
print(f'{N_TRAIN} cases pre-tokenized | max_len={max_len} | {time.time()-t0:.1f}s')
print(f'Train tensors: {TRAIN_IDS.shape} on {TRAIN_IDS.device} | {TRAIN_IDS.element_size()*TRAIN_IDS.nelement()/1e6:.0f}MB')


In [ ]:
# Cell 08 — Base Model Evaluation
gc.collect(); torch.cuda.empty_cache()
print('='*60,flush=True)
print('BASE MODEL EVALUATION',flush=True)
print('='*60,flush=True)
BASE_TARGET_ACC,BASE_TARGET_DF=evaluate_target(model,BENCHMARK_DF,tag='base_model')
BASE_CONTROL_ACC,BASE_CONTROL_DF=evaluate_control(model,BENCHMARK_DF,tag='base_model')
atomic_csv(BASE_TARGET_DF,RESULTS/'base_gpqa.csv')
atomic_csv(BASE_CONTROL_DF,RESULTS/'base_mmlu.csv')
print(f'\nBASE: GPQA={BASE_TARGET_ACC*100:.1f}% | MMLU_ctrl={BASE_CONTROL_ACC*100:.1f}%')


In [ ]:
# Cell 09 — Three-Arm Experiment (GPU-optimized batched SFT)
from peft import LoraConfig, get_peft_model

# Create SINGLE PEFT model (reuse across all runs)
print('Creating PEFT adapter...',flush=True)
for p in model.parameters(): p.requires_grad=False
peft_cfg=LoraConfig(r=LORA_RANK,lora_alpha=LORA_ALPHA,target_modules=LORA_TARGET_MODULES,
    layers_to_transform=STRATIFIED_LAYERS,bias='none',task_type='CAUSAL_LM')
peft_model=get_peft_model(model,peft_cfg)
TOTAL_LORA_PARAMS=sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
print(f'Total LoRA params: {TOTAL_LORA_PARAMS:,}',flush=True)

def reset_standard():
    for n,m in peft_model.named_modules():
        if hasattr(m,'lora_A'):
            sA=m.lora_A['default'] if hasattr(m.lora_A,'__getitem__') else m.lora_A
            sB=m.lora_B['default'] if hasattr(m.lora_B,'__getitem__') else m.lora_B
            with torch.no_grad(): nn.init.kaiming_uniform_(sA.weight,a=math.sqrt(5)); nn.init.zeros_(sB.weight)
            sA.weight.requires_grad=True; sB.weight.requires_grad=True

def reset_geodesic():
    ap=0
    for n,m in peft_model.named_modules():
        if hasattr(m,'lora_A'):
            sA=m.lora_A['default'] if hasattr(m.lora_A,'__getitem__') else m.lora_A
            sB=m.lora_B['default'] if hasattr(m.lora_B,'__getitem__') else m.lora_B
            mt=re.search(r'layers\.(\d+)\.',n); matched=False
            if mt:
                li=int(mt.group(1))
                for mn in LORA_TARGET_MODULES:
                    if mn in n and (li,mn) in WHITENED_BASES:
                        As=WHITENED_BASES[(li,mn)]
                        if sA.weight.shape==As.shape:
                            with torch.no_grad(): sA.weight.copy_(As.to(device=sA.weight.device,dtype=sA.weight.dtype)); sB.weight.zero_()
                            sA.weight.requires_grad=False  # FROZEN
                            sB.weight.requires_grad=True
                            ap+=1; matched=True
            if not matched:
                with torch.no_grad(): nn.init.kaiming_uniform_(sA.weight,a=math.sqrt(5)); nn.init.zeros_(sB.weight)
                sA.weight.requires_grad=True; sB.weight.requires_grad=True
    return ap

def reset_warm_lora():
    ap=0
    for n,m in peft_model.named_modules():
        if hasattr(m,'lora_A'):
            sA=m.lora_A['default'] if hasattr(m.lora_A,'__getitem__') else m.lora_A
            sB=m.lora_B['default'] if hasattr(m.lora_B,'__getitem__') else m.lora_B
            mt=re.search(r'layers\.(\d+)\.',n); matched=False
            if mt:
                li=int(mt.group(1))
                for mn in LORA_TARGET_MODULES:
                    if mn in n and (li,mn) in WHITENED_BASES:
                        As=WHITENED_BASES[(li,mn)]
                        if sA.weight.shape==As.shape:
                            with torch.no_grad(): sA.weight.copy_(As.to(device=sA.weight.device,dtype=sA.weight.dtype)); sB.weight.zero_()
                            sA.weight.requires_grad=True  # TRAINABLE
                            sB.weight.requires_grad=True
                            ap+=1; matched=True
            if not matched:
                with torch.no_grad(): nn.init.kaiming_uniform_(sA.weight,a=math.sqrt(5)); nn.init.zeros_(sB.weight)
                sA.weight.requires_grad=True; sB.weight.requires_grad=True
    return ap

# GPU-optimized batched SFT
def run_batched_sft(seed):
    tp=[p for p in peft_model.parameters() if p.requires_grad]
    tp_count=sum(p.numel() for p in tp)
    opt=torch.optim.AdamW(tp,lr=TRAIN_LR,betas=(0.9,0.95),weight_decay=0.01)
    rng=np.random.default_rng(int(seed))
    peft_model.train(); opt.zero_grad(set_to_none=True)
    t0=time.time(); step=0
    print(f'   SFT: {TRAIN_STEPS} steps, batch={TRAIN_BATCH}, {tp_count:,} params',flush=True)
    for epoch in range(TRAIN_EPOCHS):
        order=rng.permutation(N_TRAIN)
        for bi in range(0,N_TRAIN,TRAIN_BATCH):
            idx=order[bi:bi+TRAIN_BATCH]
            if len(idx)<2: continue
            ids=TRAIN_IDS[idx]; mask=TRAIN_MASK[idx]; labels=TRAIN_LABELS[idx]
            with torch.autocast('cuda',dtype=torch.bfloat16):
                out=peft_model(input_ids=ids,attention_mask=mask,use_cache=False)
                logits=out.logits.float()
                loss=F.cross_entropy(logits[:,:-1,:].reshape(-1,logits.shape[-1]),labels[:,1:].reshape(-1),ignore_index=-100)
            if not torch.isfinite(loss): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(tp,1.0)
            prog=step/max(1,TRAIN_STEPS-1)
            clr=LR_MIN+0.5*(TRAIN_LR-LR_MIN)*(1+math.cos(math.pi*prog))
            for pg in opt.param_groups: pg['lr']=clr
            opt.step(); opt.zero_grad(set_to_none=True)
            step+=1
            if step%32==0 or step==TRAIN_STEPS:
                print(f'      [{step:03d}/{TRAIN_STEPS}] Loss:{float(loss.detach()):.4f} LR:{clr:.2e} | {time.time()-t0:.0f}s',flush=True)
            if step>=TRAIN_STEPS: break
        if step>=TRAIN_STEPS: break
    del opt,tp; gc.collect(); torch.cuda.empty_cache()
    print(f'   Done in {time.time()-t0:.0f}s',flush=True)

# Three-arm experiment
arms=[
    ('geodesic','Geodesic (A₀ frozen)',reset_geodesic),
    ('warm_lora','Warm LoRA (A₀ trainable)',reset_warm_lora),
    ('standard','Standard LoRA (random A)',reset_standard),
]

all_results=[]
print('\n'+'='*80,flush=True)
print(f'THREE-ARM EXPERIMENT: {len(arms)} arms × {len(SEEDS)} seeds = {len(arms)*len(SEEDS)} runs',flush=True)
print('='*80,flush=True)

run_idx=0
for arm_name,arm_label,reset_fn in arms:
    for seed in SEEDS:
        run_idx+=1
        tag=f'{arm_name}_s{seed}'
        print(f'\n{"="*80}',flush=True)
        print(f'[{run_idx}/{len(arms)*len(SEEDS)}] {arm_label} | Seed {seed}',flush=True)
        print('='*80,flush=True)
        tr=time.time()
        n=reset_fn()
        tp_now=sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
        print(f'   Reset: {n if isinstance(n,int) else "ok"} modules | {tp_now:,} trainable',flush=True)
        run_batched_sft(seed)
        peft_model.eval()
        # Evaluate GPQA (target)
        t_acc,t_df=evaluate_target(peft_model,BENCHMARK_DF,tag=tag)
        atomic_csv(t_df,RESULTS/f'{tag}_gpqa.csv')
        # Evaluate MMLU (control)
        c_acc,c_df=evaluate_control(peft_model,BENCHMARK_DF,tag=tag)
        atomic_csv(c_df,RESULTS/f'{tag}_mmlu.csv')
        t_gain=t_acc-BASE_TARGET_ACC; c_drop=BASE_CONTROL_ACC-c_acc
        wall=time.time()-tr
        print(f'   RESULT: GPQA={t_acc*100:.1f}% (gain={t_gain:+.1%}) | MMLU_ctrl={c_acc*100:.1f}% (drop={c_drop:+.1%}) | {wall:.0f}s',flush=True)
        all_results.append({'arm':arm_name,'label':arm_label,'seed':seed,
            'gpqa_acc':t_acc,'gpqa_gain':t_gain,'mmlu_acc':c_acc,'mmlu_drop':c_drop,'wall_s':wall})

RESULTS_DF=pd.DataFrame(all_results)
atomic_csv(RESULTS_DF,RESULTS/'v23_all_results.csv')
print(f'\nALL {len(all_results)} RUNS COMPLETE!',flush=True)
display(RESULTS_DF)


In [ ]:
# Cell 10 — Bootstrap Analysis + Final Report
from IPython.display import display, Markdown

def bootstrap_arm(values,B=5000,seed=2026):
    rng=np.random.default_rng(seed)
    samples=[np.mean(rng.choice(values,len(values),replace=True)) for _ in range(B)]
    return {'mean':np.mean(values),'ci_lo':np.percentile(samples,2.5),'ci_hi':np.percentile(samples,97.5),'std':np.std(values)}

rows=[]
for arm,grp in RESULTS_DF.groupby('arm'):
    g_boot=bootstrap_arm(grp['gpqa_gain'].values)
    c_boot=bootstrap_arm(grp['mmlu_drop'].values)
    rows.append({'arm':arm,'label':grp['label'].iloc[0],'n':len(grp),
        'gpqa_gain_mean':g_boot['mean'],'gpqa_ci':f"[{g_boot['ci_lo']:+.3f},{g_boot['ci_hi']:+.3f}]",
        'mmlu_drop_mean':c_boot['mean'],'mmlu_ci':f"[{c_boot['ci_lo']:+.3f},{c_boot['ci_hi']:+.3f}]",
        'gpqa_gain_std':g_boot['std'],'mmlu_drop_std':c_boot['std']})

SUMMARY=pd.DataFrame(rows)
atomic_csv(SUMMARY,RESULTS/'v23_summary.csv')

md=f'# v23 Three-Arm Results\n\n'
md+=f'Base: GPQA={BASE_TARGET_ACC*100:.1f}% | MMLU_ctrl={BASE_CONTROL_ACC*100:.1f}%\n\n'
md+='| Arm | GPQA Gain | 95% CI | MMLU Drop | 95% CI | Stable? |\n'
md+='|---|:---:|:---:|:---:|:---:|:---:|\n'
for _,r in SUMMARY.iterrows():
    stable='✅' if r['gpqa_gain_std']<0.03 else '⚠️'
    md+=f'| {r["label"]} | {r["gpqa_gain_mean"]:+.3f} | {r["gpqa_ci"]} | {r["mmlu_drop_mean"]:+.3f} | {r["mmlu_ci"]} | {stable} |\n'
md+='\n'
md+='**Key comparison**: Geodesic vs Warm LoRA — same init, only difference is frozen A.\n'
md+='If Geodesic has lower MMLU drop at similar GPQA gain → constraint validated.\n'

with open(RESULTS/'v23_report.md','w') as f: f.write(md)
display(Markdown(md))
display(RESULTS_DF)
print(f'\nAll results saved to: {RESULTS}')


In [ ]:
# Cell 11 — Manifest
for f in ['benchmark.csv','base_gpqa.csv','base_mmlu.csv','v23_all_results.csv','v23_summary.csv']:
    assert (RESULTS/f).exists(), f'Missing: {f}'
print('v23 COMPLETE. All files verified.')
print(f'Results: {RESULTS}')
